In [2]:
# Imports & API Key Setup


import os
import warnings
warnings.filterwarnings("ignore")

os.environ["GOOGLE_API_KEY"] = "API"

from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain.schema import Document         
import wikipedia

print("✅ All imports successful")

✅ All imports successful


In [3]:
# Build Knowledge Base from Wikipedia

print("Loading Wikipedia articles as knowledge base...")

topics = [
    "Artificial intelligence",
    "Machine learning", 
    "Deep learning",
    "Natural language processing",
    "Neural network"
]

raw_docs = []

for topic in topics:
    try:
        page = wikipedia.page(topic, auto_suggest=False)
        content = page.content[:3000]
        raw_docs.append(Document(
            page_content=content,
            metadata={"source": topic, "url": page.url}
        ))
        print(f"  ✅ Loaded: {topic}")
    except Exception as e:
        print(f" Skipped {topic}: {e}")

print(f"\n Total documents loaded: {len(raw_docs)}")

Loading Wikipedia articles as knowledge base...
 Skipped Artificial intelligence: name 'wikipedia' is not defined
 Skipped Machine learning: name 'wikipedia' is not defined
 Skipped Deep learning: name 'wikipedia' is not defined
 Skipped Natural language processing: name 'wikipedia' is not defined
 Skipped Neural network: name 'wikipedia' is not defined

 Total documents loaded: 0


In [4]:
# Split Documents into Chunks

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " "]
)

chunks = text_splitter.split_documents(raw_docs)

print(f"✅ Total chunks created: {len(chunks)}")
print(f"\n📌 Example chunk:")
print("-" * 60)
print(chunks[0].page_content)
print(f"\nSource: {chunks[0].metadata['source']}")

✅ Total chunks created: 45

📌 Example chunk:
------------------------------------------------------------
Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.

Source: Artificial intelligence


In [8]:
# Create Vector Store (Using HuggingFace Embeddings)


from langchain_community.embeddings import HuggingFaceEmbeddings

print("Loading embedding model — first time may take 1-2 minutes to download...")

# Free, lightweight, runs locally — no API key needed
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",   # Small, fast, accurate model
    model_kwargs={"device": "cpu"}
)

print("✅ Embedding model loaded!")
print("Creating vector store...")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

vectorstore.persist()

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print(f"✅ Vector store created with {vectorstore._collection.count()} vectors")

⚙️  Loading embedding model — first time may take 1-2 minutes to download...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model loaded!
⚙️  Creating vector store...
✅ Vector store created with 45 vectors


In [17]:
# Set Up LLM + Memory + RAG Chain


llm = ChatGoogleGenerativeAI(
    model="gemma-3-4b-it",
    temperature=0.3,
    convert_system_message_to_human=True
)

memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
    output_key="answer"
)

qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    return_source_documents=True,
    verbose=False
)

print("✅ Chatbot chain ready!")

✅ Chatbot chain ready!


In [26]:
# Test the Chatbot
def ask(question):
    print(f"\n{'='*60}")
    print(f"You: {question}")
    print('='*60)
    
    result = qa_chain.invoke({"question": question})
    
    print(f"Bot: {result['answer']}")
    
    if result.get("source_documents"):
        print(f"\nSources used:")
        seen = set()
        for doc in result["source_documents"]:
            src = doc.metadata.get("source", "Unknown")
            if src not in seen:
                print(f"   • {src}")
                seen.add(src)

# Test 1
ask("What is machine learning?")


🧑 You: What is machine learning?
🤖 Bot: Machine learning is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without explicit programming language instructions.

📚 Sources used:
   • Machine learning


In [27]:
# Test 2
ask("How is it different from deep learning?")


🧑 You: How is it different from deep learning?
🤖 Bot: Here's a breakdown of the key differences between machine learning and deep learning, based on the provided text:

*   **Hierarchy of Layers:** Deep learning utilizes a “hierarchy of layers” to transform data, creating increasingly abstract representations. Machine learning doesn’t necessarily rely on this layered approach.
*   **Approach to Learning:** Machine learning involves statistical algorithms that learn from data and generalize. Deep learning is a subdiscipline of machine learning that uses neural networks to achieve this, and has often surpassed previous machine learning methods in performance.

Essentially, deep learning is a *type* of machine learning that utilizes a specific architecture (deep neural networks) to achieve better results.

📚 Sources used:
   • Deep learning
   • Machine learning


In [28]:
# Test 3
ask("Can you give a real world example of what you just explained?")


🧑 You: Can you give a real world example of what you just explained?
🤖 Bot: Okay, here’s an example illustrating the difference between machine learning and deep learning, based on the provided context:

**Machine Learning Example: Spam Email Filtering**

Imagine a traditional machine learning approach to spam email filtering. You might feed a machine learning algorithm a dataset of emails labeled as either “spam” or “not spam.” The algorithm learns patterns – like the frequency of certain words (“free,” “urgent,” “discount”), the sender’s address, or the presence of attachments – that are associated with spam. It then uses these learned patterns to classify new, incoming emails. This is a relatively shallow process; the algorithm is essentially looking for specific, pre-defined features.

**Deep Learning Example: Image Recognition (like identifying cats in photos)**

Now, consider a deep learning approach to the same task – identifying cats in photos. A deep learning model (like a co

In [29]:
# Evaluation
import pandas as pd

print("📊 Running Evaluation...\n")

test_questions = [
    "What is a neural network?",
    "Explain NLP in simple terms",
    "What are the applications of deep learning?"
]

results = []

for q in test_questions:
    result = qa_chain.invoke({"question": q})
    sources = list(set([
        d.metadata.get("source", "Unknown")
        for d in result.get("source_documents", [])
    ]))
    results.append({
        "Question": q,
        "Answer Length (chars)": len(result["answer"]),
        "Sources Used": ", ".join(sources) if sources else "None"
    })
    print(f"✅ Done: {q[:50]}")

df = pd.DataFrame(results)
print("\n📋 Evaluation Table:")
print(df.to_string(index=False))

📊 Running Evaluation...

✅ Done: What is a neural network?
✅ Done: Explain NLP in simple terms
✅ Done: What are the applications of deep learning?

📋 Evaluation Table:
                                   Question  Answer Length (chars)                  Sources Used
                  What is a neural network?                    456                Neural network
                Explain NLP in simple terms                    385   Natural language processing
What are the applications of deep learning?                    197 Deep learning, Neural network


In [31]:
# Save Streamlit App
streamlit_code = '''
import streamlit as st
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain.prompts import PromptTemplate

st.set_page_config(page_title="RAG Chatbot", page_icon="🤖", layout="centered")
st.title("RAG Chatbot")
st.caption("Powered by Gemma 3 + LangChain + ChromaDB + HuggingFace Embeddings")

api_key = st.sidebar.text_input("Gemini API Key", type="password")
if not api_key:
    st.warning("Please enter your Gemini API key in the sidebar to start.")
    st.stop()

os.environ["GOOGLE_API_KEY"] = api_key

@st.cache_resource
def load_chain():
    embeddings = HuggingFaceEmbeddings(
        model_name="all-MiniLM-L6-v2",
        model_kwargs={"device": "cpu"}
    )
    vectorstore = Chroma(
        persist_directory="./chroma_db",
        embedding_function=embeddings
    )
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

    llm = ChatGoogleGenerativeAI(
        model="gemma-3-4b-it",
        temperature=0.3,
        convert_system_message_to_human=True
    )
    memory = ConversationBufferMemory(
        memory_key="chat_history",
        return_messages=True,
        output_key="answer"
    )

    condense_template = """Given the following conversation and a follow-up question, rephrase the follow-up question to be a standalone question.

Chat History:
{chat_history}
Follow Up Input: {question}
Standalone question:"""

    answer_template = """Use the following pieces of context to answer the question at the end.
If you do not know the answer, just say that you do not know.

{context}

Question: {question}
Helpful Answer:"""

    chain = ConversationalRetrievalChain.from_llm(
        llm=llm,
        retriever=retriever,
        memory=memory,
        condense_question_prompt=PromptTemplate.from_template(condense_template),
        combine_docs_chain_kwargs={"prompt": PromptTemplate.from_template(answer_template)},
        return_source_documents=True,
        get_chat_history=lambda h: h,
    )
    return chain

chain = load_chain()

if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

if prompt := st.chat_input("Ask me anything about AI/ML..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            result = chain.invoke({"question": prompt})
            answer = result["answer"]
            sources = list(set([
                d.metadata.get("source", "Unknown")
                for d in result.get("source_documents", [])
            ]))
        st.markdown(answer)
        if sources:
            st.caption("Sources: " + ", ".join(sources))

    st.session_state.messages.append({"role": "assistant", "content": answer})
'''

with open("app.py", "w", encoding="utf-8") as f:  
    f.write(streamlit_code)

print("app.py saved successfully!")
print("\nTo launch Streamlit, open a terminal and run:")
print("    streamlit run app.py")

app.py saved successfully!

To launch Streamlit, open a terminal and run:
    streamlit run app.py
